In [1]:
import pandas as pd

ml_table = pd.read_csv("ml_table.csv")

ml_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,total_price,total_freight,payment_count,total_payment,max_installments,review_count,avg_review_score,unique_products,unique_sellers,unique_categories
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,3.0,38.71,1.0,1.0,4.0,1.0,1.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,1.0,141.46,1.0,1.0,4.0,1.0,1.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,1.0,179.12,3.0,1.0,5.0,1.0,1.0,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,1.0,72.20,1.0,1.0,5.0,1.0,1.0,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,1.0,28.62,1.0,1.0,5.0,1.0,1.0,1.0


In [2]:
# Convert delivery dates to datetime
ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"],
    errors="coerce"
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"],
    errors="coerce"
)

# Keep orders where we know both dates
labeled_df = ml_table.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
).copy()

# 1 = Late, 0 = On-time
labeled_df["is_late"] = (
    labeled_df["order_delivered_customer_date"]
    > labeled_df["order_estimated_delivery_date"]
).astype(int)

print("Rows with valid delivery dates:", len(labeled_df))
print("\nLabel distribution:")
print(labeled_df["is_late"].value_counts())

print("\nPercentages:")
print(labeled_df["is_late"].value_counts(normalize=True) * 100)

Rows with valid delivery dates: 96476

Label distribution:
is_late
0    88649
1     7827
Name: count, dtype: int64

Percentages:
is_late
0    91.887101
1     8.112899
Name: proportion, dtype: float64


In [3]:
# Sanity check
print(
    labeled_df[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "is_late"
        ]
    ].head(10)
)

# Save artifact for the next notebook
labeled_df.to_csv("labeled_table.csv", index=False)

print("\nlabeled_table.csv saved successfully!")
print("Final shape:", labeled_df.shape)

   order_delivered_customer_date order_estimated_delivery_date  is_late
0            2017-10-10 21:25:13                    2017-10-18        0
1            2018-08-07 15:27:45                    2018-08-13        0
2            2018-08-17 18:06:29                    2018-09-04        0
3            2017-12-02 00:28:42                    2017-12-15        0
4            2018-02-16 18:17:02                    2018-02-26        0
5            2017-07-26 10:57:55                    2017-08-01        0
7            2017-05-26 12:55:51                    2017-06-07        0
8            2017-02-02 14:08:10                    2017-03-06        0
9            2017-08-16 17:14:30                    2017-08-23        0
10           2017-05-29 11:18:31                    2017-06-07        0

labeled_table.csv saved successfully!
Final shape: (96476, 24)
